# Tutorial 11: HLLSet De Bruijn — Evolution Analysis & Token Recovery

**Tier 3 — Disambiguation & Reconstruction**

Tutorial 09 introduced D/R/N triples and the HLLSet De Bruijn graph.
This tutorial goes deeper: we **recover tokens** from each D/R/N component
and **classify the evolution dynamics** of HLLSet sequences.

## What You'll Learn

1. **Token Recovery from D/R/N** — Invert each component back to ordered tokens
2. **Evolution Phase Classification** — Growth, Decay, Stable, Replacement
3. **Evolution Summary** — Aggregate statistics over an entire path
4. **Multi-Version Document Analysis** — End-to-end example

## Prerequisites

| Tutorial | Concept |
|----------|---------|
| [09_hllset_debruijn.ipynb](09_hllset_debruijn.ipynb) | D/R/N triples, HLLSet De Bruijn graph, path finding |
| [10_disambiguation.ipynb](10_disambiguation.ipynb) | DisambiguationEngine, token recovery |
| [07_bss.ipynb](07_bss.ipynb) | Bell State Similarity (τ, ρ) |

## Architecture

```
HLLSet H₁ ─────────────── HLLSet H₂
     │   decompose_transition    │
     └──────────┬────────────────┘
                ▼
        DRNTriple (D, R, N)
                │
      ┌─────────┼─────────┐
      ▼         ▼         ▼
   Deleted   Retained   Novel
   tokens    tokens     tokens
      │         │         │
      ▼         ▼         ▼
 DisambiguationEngine per component
      │         │         │
      └─────────┼─────────┘
                ▼
        Token Recovery
        (De Bruijn path)
```

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

from core.hllset_debruijn import (
    DRNTriple,
    FullDRNTriple,
    HLLSetDeBruijnGraph,
    decompose_transition,
    full_decompose_transition,
    verify_reconstruction,
    build_hllset_debruijn,
    find_evolution_path,
    recover_tokens_from_drn,
    EvolutionPhase,
    classify_transition,
    EvolutionSummary,
    analyze_evolution,
)
from core.hllset import HLLSet
from core.bss import bss

P_BITS = 10

## 1. Full D/R/N Decomposition with Tokens

`full_decompose_transition` preserves the **original token order** within each
D/R/N component — this is the prerequisite for De Bruijn recovery.

In [ ]:
# Two document versions
tokens_v1 = ["the", "quick", "brown", "fox", "jumps", "over", "the", "lazy", "dog"]
tokens_v2 = ["the", "fast", "brown", "fox", "leaps", "over", "the", "sleepy", "cat"]

# Full decomposition preserves token order
full_drn = full_decompose_transition(tokens_v1, tokens_v2, p_bits=P_BITS)

print("Full D/R/N Decomposition (v1 → v2):")
print(f"  Deleted:  {full_drn.deleted_tokens}")
print(f"  Retained: {full_drn.retained_tokens}")
print(f"  Novel:    {full_drn.novel_tokens}")

# Each component is also an HLLSet
print(f"\nHLLSet cardinalities:")
print(f"  |D| ≈ {full_drn.deleted_hll.cardinality():.0f}")
print(f"  |R| ≈ {full_drn.retained_hll.cardinality():.0f}")
print(f"  |N| ≈ {full_drn.novel_hll.cardinality():.0f}")

In [ ]:
# Access the basic DRN triple (HLLSets only, no tokens)
basic = full_drn.drn
print(f"Basic DRN triple type: {type(basic).__name__}")
print(f"  D is_growth: {basic.is_growth()}")
print(f"  net_change:  {basic.net_change():.0f}")

## 2. Token Recovery from D/R/N

`recover_tokens_from_drn` creates a **DisambiguationEngine per component**,
ingests n-grams with START/END markers, then runs De Bruijn path recovery.

This demonstrates that HLLSet transitions are **fully invertible**.

In [ ]:
# Recover tokens from each D/R/N component via De Bruijn
recovery = recover_tokens_from_drn(full_drn, p_bits=P_BITS)

for component, info in recovery.items():
    original = info["original"]
    recovered = info["recovered"]
    match = info["match"]
    print(f"{component} component:")
    print(f"  Original:  {original}")
    print(f"  Recovered: {recovered}")
    print(f"  Match: {match}")
    print()

In [ ]:
# Verify: recovered tokens should reconstruct the same transition
# Original: v1 has (deleted + retained), v2 has (retained + novel)
d_tokens = recovery["D"]["recovered"]
r_tokens = recovery["R"]["recovered"]
n_tokens = recovery["N"]["recovered"]

print("Round-trip verification:")
print(f"  v1 ≈ D ∪ R = {set(d_tokens) | set(r_tokens)}")
print(f"  v2 ≈ R ∪ N = {set(r_tokens) | set(n_tokens)}")
print(f"\n  Original v1 tokens: {set(tokens_v1)}")
print(f"  Original v2 tokens: {set(tokens_v2)}")

## 3. Evolution Phase Classification

Every transition falls into one of four phases:

| Phase | Condition | Meaning |
|-------|-----------|---------|
| **GROWTH** | \|N\| > \|D\| | System expanding |
| **DECAY** | \|D\| > \|N\| | System shrinking |
| **STABLE** | \|N\| ≈ \|D\| | Balanced exchange |
| **REPLACEMENT** | \|R\| < 0.5(\|D\|+\|N\|) | Low retention, high turnover |

In [ ]:
# Create transitions with different characteristics
scenarios = {
    "Growth": (
        ["a", "b", "c"],
        ["a", "b", "c", "d", "e"]
    ),
    "Decay": (
        ["a", "b", "c", "d", "e"],
        ["a", "b"]
    ),
    "Stable": (
        ["a", "b", "c"],
        ["b", "c", "d"]
    ),
    "Replacement": (
        ["a", "b", "c"],
        ["x", "y", "z"]
    ),
}

print("Evolution Phase Classification:")
print(f"{'Scenario':<15} {'|D|':>5} {'|R|':>5} {'|N|':>5} {'Phase':<15}")
print("-" * 55)

for name, (t1, t2) in scenarios.items():
    h1 = HLLSet.from_batch(t1, p_bits=P_BITS)
    h2 = HLLSet.from_batch(t2, p_bits=P_BITS)
    drn = decompose_transition(h1, h2)
    phase = classify_transition(drn)
    print(f"{name:<15} {drn.deleted_card:5.0f} {drn.retained_card:5.0f} {drn.novel_card:5.0f} {phase.value:<15}")

In [ ]:
# The stability_threshold parameter controls the STABLE boundary
drn_balanced = decompose_transition(
    HLLSet.from_batch(["a", "b", "c"], p_bits=P_BITS),
    HLLSet.from_batch(["b", "c", "d"], p_bits=P_BITS),
)

# Default threshold = 0.1
phase_default = classify_transition(drn_balanced, stability_threshold=0.1)
# Tight threshold = 0.01
phase_tight = classify_transition(drn_balanced, stability_threshold=0.01)

print(f"Balanced transition (|D|≈{drn_balanced.deleted_card:.0f}, |N|≈{drn_balanced.novel_card:.0f}):")
print(f"  threshold=0.1  → {phase_default.value}")
print(f"  threshold=0.01 → {phase_tight.value}")

## 4. Building an Evolution Sequence

Let's create a realistic document evolution and analyze the full trajectory.

In [ ]:
# Simulate document evolution: a research paper draft
versions = [
    "machine learning models process data",                       # v0: initial draft
    "deep learning models process large data efficiently",        # v1: expanded
    "deep learning neural networks process large data sets",      # v2: terminology shift
    "deep neural networks train on large data sets quickly",      # v3: more changes
    "advanced neural networks train on massive data sets",        # v4: final polish
]

version_labels = [f"v{i}" for i in range(len(versions))]

# Create HLLSets
states = [HLLSet.from_batch(v.split(), p_bits=P_BITS) for v in versions]

print("Document versions:")
for label, text, hll in zip(version_labels, versions, states):
    print(f"  {label}: '{text}'  (|H|≈{hll.cardinality():.0f})")

In [ ]:
# Build HLLSet De Bruijn graph
graph = build_hllset_debruijn(
    states,
    labels=version_labels,
    tau_min=0.2,
    rho_max=0.8,
    p_bits=P_BITS,
)

print(f"HLLSet De Bruijn Graph:")
print(f"  Nodes: {graph.num_nodes}")
print(f"  Edges: {graph.num_edges}")
print(f"  τ_min: {graph.tau_min}, ρ_max: {graph.rho_max}")

In [ ]:
# Find evolution path from the (possibly shuffled) graph
path = find_evolution_path(graph)

recovered_order = [version_labels[i] for i in path]
print(f"Recovered evolution path: {' → '.join(recovered_order)}")
print(f"Expected:                 v0 → v1 → v2 → v3 → v4")

## 5. Evolution Summary

`analyze_evolution` aggregates D/R/N statistics and phase classifications
over the entire path, producing an `EvolutionSummary`.

In [ ]:
# Analyze evolution along the recovered path
summary = analyze_evolution(graph, path)

print(f"Evolution Summary:")
print(f"  Path length: {len(summary.path)} nodes, {len(summary.transitions)} transitions")
print(f"  Total deleted:  {summary.total_deleted:.0f}")
print(f"  Total novel:    {summary.total_novel:.0f}")
print(f"  Net growth:     {summary.net_growth:+.0f}")
print(f"  Avg retained:   {summary.total_retained_avg:.1f}")
print(f"  Dominant phase: {summary.dominant_phase.value}")

In [ ]:
# Phase breakdown per transition
print(f"\nPer-transition breakdown:")
print(f"{'Step':<10} {'From':>4} {'To':>4} {'|D|':>5} {'|R|':>5} {'|N|':>5} {'Phase':<12}")
print("-" * 55)

for i, (drn, phase) in enumerate(zip(summary.transitions, summary.phases)):
    src_label = version_labels[summary.path[i]]
    tgt_label = version_labels[summary.path[i + 1]]
    print(
        f"  {i+1:<6} {src_label:>4} → {tgt_label:<4} "
        f"{drn.deleted_card:5.0f} {drn.retained_card:5.0f} {drn.novel_card:5.0f} "
        f"{phase.value:<12}"
    )

## 6. Token Recovery Along the Evolution Path

For each transition in the path, we can recover the **exact tokens** that were
deleted, retained, and added — making the evolution fully transparent.

In [ ]:
# Recover tokens for each consecutive transition
print("Token-level evolution trace:")
for i in range(len(path) - 1):
    src_idx, tgt_idx = path[i], path[i + 1]
    src_tokens = versions[src_idx].split()
    tgt_tokens = versions[tgt_idx].split()

    full_drn = full_decompose_transition(src_tokens, tgt_tokens, p_bits=P_BITS)
    recovery = recover_tokens_from_drn(full_drn, p_bits=P_BITS)

    src_label = version_labels[src_idx]
    tgt_label = version_labels[tgt_idx]
    print(f"\n  {src_label} → {tgt_label}:")
    print(f"    Deleted:  {recovery['D']['recovered']}")
    print(f"    Retained: {recovery['R']['recovered']}")
    print(f"    Novel:    {recovery['N']['recovered']}")
    print(f"    All matched: D={recovery['D']['match']}, R={recovery['R']['match']}, N={recovery['N']['match']}")

## 7. Reconstruction Verification

The fundamental algebraic invariant: $H_2 = (H_1 \setminus D) \cup R \cup N$

Verify this holds for every transition in the path.

In [ ]:
# Verify reconstruction for every step
print("Algebraic reconstruction verification:")
all_valid = True
for i in range(len(path) - 1):
    src_idx, tgt_idx = path[i], path[i + 1]
    h1, h2 = states[src_idx], states[tgt_idx]
    drn = decompose_transition(h1, h2)

    valid = verify_reconstruction(h1, h2, drn, tolerance=2.0)
    all_valid &= valid

    reconstructed = h1.diff(drn.deleted).union(drn.retained).union(drn.novel)
    diff = abs(reconstructed.cardinality() - h2.cardinality())

    src_label = version_labels[src_idx]
    tgt_label = version_labels[tgt_idx]
    print(f"  {src_label}→{tgt_label}: |H₂|≈{h2.cardinality():.1f}, |recon|≈{reconstructed.cardinality():.1f}, Δ={diff:.2f}, valid={valid}")

print(f"\nAll transitions valid: {all_valid}")

## 8. DOT Visualization with Phase Labels

In [ ]:
# Generate DOT with evolution phase annotations
dot = graph.to_dot(title="DocumentEvolution")
print(dot)

In [ ]:
# Enhanced DOT with phase colors
def evolution_dot(graph, labels, path):
    """Generate DOT graph with evolution phase coloring."""
    lines = [
        'digraph Evolution {',
        '  rankdir=LR;',
        '  node [shape=ellipse, style=filled, fillcolor=lightyellow];',
    ]
    # Nodes
    for i, (hll, label) in enumerate(zip(graph.nodes, labels)):
        card = hll.cardinality()
        lines.append(f'  H{i} [label="{label}\\n|H|≈{card:.0f}"];')

    # Color edges by phase
    phase_colors = {
        EvolutionPhase.GROWTH: "green",
        EvolutionPhase.DECAY: "red",
        EvolutionPhase.STABLE: "blue",
        EvolutionPhase.REPLACEMENT: "orange",
    }

    for edge in graph.edges:
        drn = edge.drn
        phase = classify_transition(drn)
        color = phase_colors.get(phase, "black")
        label = f"{phase.value}\\nΔ={drn.net_change():+.0f}"
        lines.append(
            f'  H{edge.source_idx} -> H{edge.target_idx} '
            f'[label="{label}", color="{color}", fontcolor="{color}"];'
        )

    lines.append('}')
    return '\n'.join(lines)

print(evolution_dot(graph, version_labels, path))

## 9. Comparing Evolution Patterns

Create two different evolution trajectories and compare their summaries.

In [ ]:
# Trajectory A: Steady growth
growth_texts = [
    "data",
    "data science",
    "data science machine learning",
    "data science machine learning deep networks",
]
growth_states = [HLLSet.from_batch(t.split(), p_bits=P_BITS) for t in growth_texts]
growth_graph = build_hllset_debruijn(growth_states, tau_min=0.1, rho_max=0.9, p_bits=P_BITS)
growth_path = find_evolution_path(growth_graph)
growth_summary = analyze_evolution(growth_graph, growth_path)

# Trajectory B: Replacement-heavy
replace_texts = [
    "alpha beta gamma",
    "delta epsilon zeta",
    "eta theta iota",
    "kappa lambda mu",
]
replace_states = [HLLSet.from_batch(t.split(), p_bits=P_BITS) for t in replace_texts]
replace_graph = build_hllset_debruijn(replace_states, tau_min=0.0, rho_max=1.0, p_bits=P_BITS)
replace_path = find_evolution_path(replace_graph)
replace_summary = analyze_evolution(replace_graph, replace_path)

print(f"{'Metric':<25} {'Growth Traj':>15} {'Replacement Traj':>18}")
print("-" * 60)
print(f"{'Total deleted':<25} {growth_summary.total_deleted:>15.0f} {replace_summary.total_deleted:>18.0f}")
print(f"{'Total novel':<25} {growth_summary.total_novel:>15.0f} {replace_summary.total_novel:>18.0f}")
print(f"{'Net growth':<25} {growth_summary.net_growth:>+15.0f} {replace_summary.net_growth:>+18.0f}")
print(f"{'Avg retained':<25} {growth_summary.total_retained_avg:>15.1f} {replace_summary.total_retained_avg:>18.1f}")
print(f"{'Dominant phase':<25} {growth_summary.dominant_phase.value:>15} {replace_summary.dominant_phase.value:>18}")

In [ ]:
# Phase distribution comparison
from collections import Counter

for name, summary in [("Growth trajectory", growth_summary), ("Replacement trajectory", replace_summary)]:
    counts = Counter(summary.phases)
    print(f"\n{name} phase distribution:")
    for phase in EvolutionPhase:
        count = counts.get(phase, 0)
        bar = "█" * count
        print(f"  {phase.value:<15} {count:>2} {bar}")

## 10. Edge Cases: Empty Components

When a transition has no deletions or no additions, the corresponding
D/R/N component is an empty HLLSet. Token recovery handles this gracefully.

In [ ]:
# Pure addition: no deletions
tokens_base = ["hello", "world"]
tokens_expanded = ["hello", "world", "foo", "bar"]

full_drn = full_decompose_transition(tokens_base, tokens_expanded, p_bits=P_BITS)
recovery = recover_tokens_from_drn(full_drn, p_bits=P_BITS)

print("Pure addition (no deletions):")
print(f"  D: {recovery['D']['original']}  → recovered: {recovery['D']['recovered']}  match: {recovery['D']['match']}")
print(f"  R: {recovery['R']['original']}  → recovered: {recovery['R']['recovered']}  match: {recovery['R']['match']}")
print(f"  N: {recovery['N']['original']}  → recovered: {recovery['N']['recovered']}  match: {recovery['N']['match']}")

In [ ]:
# Pure deletion: no additions
full_drn2 = full_decompose_transition(tokens_expanded, tokens_base, p_bits=P_BITS)
recovery2 = recover_tokens_from_drn(full_drn2, p_bits=P_BITS)

print("Pure deletion (no additions):")
print(f"  D: {recovery2['D']['original']}  → recovered: {recovery2['D']['recovered']}  match: {recovery2['D']['match']}")
print(f"  R: {recovery2['R']['original']}  → recovered: {recovery2['R']['recovered']}  match: {recovery2['R']['match']}")
print(f"  N: {recovery2['N']['original']}  → recovered: {recovery2['N']['recovered']}  match: {recovery2['N']['match']}")

## 11. Information Flux Profile

Plot the information flux $\Phi(t) = |N(t)| - |D(t)|$ across the evolution.

In [ ]:
# Compute flux profile for the document evolution
print("Information flux Φ(t) = |N| - |D|:")
print()

for i in range(len(path) - 1):
    src_idx, tgt_idx = path[i], path[i + 1]
    drn = decompose_transition(states[src_idx], states[tgt_idx])
    flux = drn.net_change()

    src_label = version_labels[src_idx]
    tgt_label = version_labels[tgt_idx]

    # ASCII bar chart
    bar_len = int(abs(flux))
    if flux > 0:
        bar = " " * 10 + "│" + "█" * bar_len
    elif flux < 0:
        bar = " " * max(0, 10 - bar_len) + "█" * bar_len + "│"
    else:
        bar = " " * 10 + "│"

    print(f"  {src_label}→{tgt_label}: Φ={flux:+5.0f}  {bar}")

print()
print(f"  Net cumulative flux: {summary.net_growth:+.0f}")

## Summary

In this tutorial, you learned:

1. **Token Recovery from D/R/N** — `recover_tokens_from_drn()` inverts each component
2. **Phase Classification** — `classify_transition()` labels Growth/Decay/Stable/Replacement
3. **Evolution Summary** — `analyze_evolution()` aggregates path-wide statistics
4. **Information Flux** — Φ(t) = |N| - |D| tracks system expansion/contraction

### API Reference

| Function | Purpose |
|----------|---------|
| `full_decompose_transition(t1, t2)` | D/R/N with ordered tokens |
| `recover_tokens_from_drn(drn)` | De Bruijn recovery per component |
| `classify_transition(drn)` | → `EvolutionPhase` enum |
| `analyze_evolution(graph, path)` | → `EvolutionSummary` dataclass |

### EvolutionSummary Properties

| Property | Description |
|----------|-------------|
| `total_deleted` | Σ |D(t)| across all transitions |
| `total_novel` | Σ |N(t)| across all transitions |
| `net_growth` | total_novel − total_deleted |
| `total_retained_avg` | Average |R(t)| per step |
| `dominant_phase` | Most frequent phase in the path |

### Connection to Tier 4

The information flux Φ(t) connects directly to the **Noether Steering Law** (Tutorial 13):

$$\text{If } |N(t)| = |D(t)| \text{ for all } t, \text{ then } \sum_i R_i(t) \text{ is conserved.}$$

### Next Steps

- **Tutorial 12**: Lattice Reconstruction — Restore W lattice from HLLSets